### Middleware (hooks)

#Middleware provides a way to more tightly control what happen inside the agent. Middleware is useful for following:
   * 1.Tracking agent behavior with logging, analytics and debugging.
   * 2.Transforming prompts, tool selection and output formatting.
   * 3.Adding retires, fallback and early terminations logic.
   * 4.Applying rate limit, guardrails and security policies.
   * 5.Managing conversation history and memory.

Example ==> Airport Process

Passenger -> Security Check -> Immigration -> Boarding Gate -> Flight 

#### Built in Middleware
* Summarization => Summarize the conversation history to reduce the number of tokens sent to the LLM. (e. after every 10 msg generate Summery)
* Human-in-the-loop => Pause execution for human approval of tool calls. Ask for human feedback after every step.
* Model call limit => Limit the number of LLM calls.
* Tool call limit => Limit the number of tool calls.
.
.
.
Their are so many LLM middleware available in the market.

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

### Summarization Middleware

Automattically summarize conversation history when approaching token limits,preserving recent message while compressing older context. Summarization is useful for following:

- Long running conversation that exceed context window
- Multi-turn dialog with extensive history
- Application where preserving full conversation context matter

In [11]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage,SystemMessage


### MessageBased Summarization
agent=create_agent(
    model="groq:qwen/qwen3.6-27b",
    checkpointer=InMemorySaver(),  ### Where you want to store your messages
    middleware=[
        SummarizationMiddleware(
            model="groq:openai/gpt-oss-120b",
            trigger=("messages",10),    ### trigger the summarization middleware after every 10 messages 
            keep=("messages",4)         ### keep last 4 messages
        )
    ]
)

In [13]:
### Run with thread ID
config={"configurable":{"thread_id":"test-1"}}

In [14]:
### Alternative test data
questions=[
    "What is 2+2?",
    "What is 4*6?",
    "What is 4/2?",
    "What is 3*3?",
    "What is 4*4?",
    "What is 35/67?"
]

for q in questions:
    response=agent.invoke({"messages":[HumanMessage(content=q)]},config)
    print(f"Messages:{response}")
    print(f"Messages:{len(response["messages"])}")

Messages:{'messages': [HumanMessage(content='What is 2+2?', additional_kwargs={}, response_metadata={}, id='68fa4e22-177c-47b2-b6be-e94aea671d47'), AIMessage(content='\n<think>\nHere\'s a thinking process:\n\n1.  **Analyze User Input:** The user asks "What is 2+2?"\n2.  **Identify Core Task:** This is a basic arithmetic question.\n3.  **Perform Calculation:** 2 + 2 = 4.\n4.  **Formulate Response:** State the answer clearly and concisely.\n5.  **Check for Accuracy:** 2+2 is universally 4 in standard arithmetic. No tricks or context suggest otherwise.\n6.  **Output Generation:** "2 + 2 equals 4." (or similar straightforward answer)✅\n</think>\n\n2 + 2 equals 4.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 143, 'prompt_tokens': 17, 'total_tokens': 160, 'completion_time': 0.272250107, 'completion_tokens_details': None, 'prompt_time': 0.000871997, 'prompt_tokens_details': None, 'queue_time': 0.052330221, 'total_time': 0.273122104}, 'model_name': 'qwen/qwen

### Token Size


In [15]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage


@tool
def search_hotels(city:str)->str:
    """Search hotels - return long response to use more tokens."""
    return f"""Hotels in {city}
    1. Grand Hotel - 5 stars 
    2. Beach Resort - 4 stars
    3. Downtown Inn - 3 stars """

In [20]:
agent=create_agent(
   model="groq:qwen/qwen3.6-27b",
   tools=[search_hotels],
    checkpointer=InMemorySaver(),
     middleware=[
            SummarizationMiddleware(
                model="groq:openai/gpt-oss-120b",
                trigger=("tokens",350),    ### trigger the summarization middleware after every 550 tokens 
                keep=("tokens",200)         ### keep last 200 tokens
            )
        ]
)

config={"configurable":{"thread_id":"test-2"}}   ### Call for the specific user

#Token counter
def count_tokens(messages):
    total_chars=sum(len(str(m.content))for m in messages)
    return total_chars // 4

In [21]:
# Run test
cities=["Paris","London","Tokyo","New York","Dubai","Singapore"]

for city in cities:
    response=agent.invoke(
        {"messages":[HumanMessage(content=f"Find Hotels in {city}")]},
        config=config
    )
    
    tokens=count_tokens(response["messages"])
    print(f"{city}: ~ {tokens} tokens,{len(response['messages'])} messages")
    print(f"{(response['messages'])}")

Paris: ~ 64 tokens,4 messages
[HumanMessage(content='Find Hotels in Paris', additional_kwargs={}, response_metadata={}, id='b504e1ed-acbc-4c95-ad04-d404b8810a60'), AIMessage(content='', additional_kwargs={'reasoning_content': 'Here\'s a thinking process:\n\n1.  **Understand User Request:** The user wants to find hotels in Paris.\n2.  **Identify Available Tools:** I have a `search_hotels` function that takes a `city` parameter.\n3.  **Match Request to Tool:** The request matches the tool perfectly. The city is "Paris".\n4.  **Execute Tool Call:** Call `search_hotels` with `city: "Paris"`.\n5.  **Process Response:** (I\'ll simulate the tool execution mentally, but I need to actually call it to get the real response. I\'ll proceed with the tool call.)\n6.  **Formulate Response:** After getting the results, I\'ll present them clearly to the user, highlighting key details like hotel names, prices, ratings, locations, and amenities. I\'ll also offer to help with filtering or booking if neede

### Fraction
 Fraction is another trigger

### Human in the Loop Middleware

Pause agent execution for human approval,editing or rejection of tool call before they execute. Human-in-the-loop is useful for the following
- High-stakes operations requiring human approval (e.g database writes,financial transactions)
- Compliance workflows where human oversight is mandatory.
- Long-running conversations where human feedback guides the agent

In [23]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

def read_email_tool(email_id:str)->str:
    """Mock function to read an email by its ID"""
    return f"Email content for ID:{email_id }"

def send_email_tool(recipient:str,subject:str,body:str)->str:
    """Mock function to send an email"""
    return f"Email sent to {recipient} with subject '{subject}'"

In [28]:
agent=create_agent(
    model="groq:openai/gpt-oss-120b",
    tools=[read_email_tool,send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool":{
                    "allowed_decisions":["approve","edit","reject"]
                },
                "read_email_tool":False
            }
        )
    ]
)

In [44]:
config={"configurable":{"thread_id":"test-approve"}}   # Thread_Id is unique Id
# step1: Request
result=agent.invoke(
    {"messages":[HumanMessage(content="Send email to john@test.com  with subject 'Hello' and Body 'How are you?'")]},
    config
)
result

{'messages': [HumanMessage(content="Send email to john@test.com  with subject 'Hello' and Body 'How are you?'", additional_kwargs={}, response_metadata={}, id='f2347dc0-239b-497c-b622-8f313c9e43d8'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'The user wants to send an email. We need to call send_email_tool with appropriate parameters.', 'tool_calls': [{'id': 'fc_3c055e20-3945-4842-bf6b-18d5a8be837c', 'function': {'arguments': '{"body":"How are you?","recipient":"john@test.com","subject":"Hello"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 66, 'prompt_tokens': 175, 'total_tokens': 241, 'completion_time': 0.141679495, 'completion_tokens_details': {'reasoning_tokens': 20}, 'prompt_time': 0.007036514, 'prompt_tokens_details': None, 'queue_time': 0.376511432, 'total_time': 0.148716009}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_017482bd7f', 'service_tier': 'on_demand', 'finish_reason'

In [ ]:
# Step 2: Approve

from langgraph.types import Command

if "__interrupt__" in result:
    print("Paused! Approving...")
    
    result = agent.invoke(
       Command(resume={"decisions": [{"type": "approve"}]}),
        config=config
    )
    
    
    print(f"Result: {result['messages'][-1].context}")

In [47]:
result

{'messages': [HumanMessage(content="Send email to john@test.com  with subject 'Hello' and Body 'How are you?'", additional_kwargs={}, response_metadata={}, id='f2347dc0-239b-497c-b622-8f313c9e43d8'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'The user wants to send an email. We need to call send_email_tool with appropriate parameters.', 'tool_calls': [{'id': 'fc_3c055e20-3945-4842-bf6b-18d5a8be837c', 'function': {'arguments': '{"body":"How are you?","recipient":"john@test.com","subject":"Hello"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 66, 'prompt_tokens': 175, 'total_tokens': 241, 'completion_time': 0.141679495, 'completion_tokens_details': {'reasoning_tokens': 20}, 'prompt_time': 0.007036514, 'prompt_tokens_details': None, 'queue_time': 0.376511432, 'total_time': 0.148716009}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_017482bd7f', 'service_tier': 'on_demand', 'finish_reason'

### Reject

In [52]:
agent=create_agent(
    model="groq:openai/gpt-oss-120b",
    tools=[read_email_tool,send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool":{
                    "allowed_decisions":["approve","edit","reject"]
                },
                "read_email_tool":False
            }
        )
    ]
)

In [53]:
config={"configurable":{"thread_id":"test-reject"}}   # Thread_Id is unique Id
# step1: Request
result=agent.invoke(
    {"messages":[HumanMessage(content="Send email to john@test.com  with subject 'Hello' and Body 'How are you?'")]},
    config
)

In [ ]:
# Step 2: Reject

from langgraph.types import Command

if "__interrupt__" in result:
    print("Paused! Approving...")
    
    result = agent.invoke(
       Command(resume={"decisions": [{"type": "reject"}]}),
        config=config
    )
    
    
    #print(f"Result: {result['messages'][-1].context}")

In [56]:
result

{'messages': [HumanMessage(content="Send email to john@test.com  with subject 'Hello' and Body 'How are you?'", additional_kwargs={}, response_metadata={}, id='9c239e9f-a8fd-4041-b3fb-7acf1e07e7c7'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'The user wants to send an email. Use send_email_tool with body, recipient, subject.', 'tool_calls': [{'id': 'fc_9d263644-bf66-4f41-92c3-4774e77d8baf', 'function': {'arguments': '{"body":"How are you?","recipient":"john@test.com","subject":"Hello"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 66, 'prompt_tokens': 175, 'total_tokens': 241, 'completion_time': 0.13730982, 'completion_tokens_details': {'reasoning_tokens': 20}, 'prompt_time': 0.025020186, 'prompt_tokens_details': None, 'queue_time': 0.382144659, 'total_time': 0.162330006}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_e1a78f200e', 'service_tier': 'on_demand', 'finish_reason': 'tool_cal

### Editing

In [57]:
agent=create_agent(
    model="groq:openai/gpt-oss-120b",
    tools=[read_email_tool,send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool":{
                    "allowed_decisions":["approve","edit","reject"]
                },
                "read_email_tool":False
            }
        )
    ]
)

In [58]:
config={"configurable":{"thread_id":"test-edit"}}   # Thread_Id is unique Id
# step1: Request
result=agent.invoke(
    {"messages":[HumanMessage(content="Send email to wrong@test.com  with subject 'Hello' and Body 'How are you?'")]},
    config
)

In [59]:
result

{'messages': [HumanMessage(content="Send email to wrong@test.com  with subject 'Hello' and Body 'How are you?'", additional_kwargs={}, response_metadata={}, id='c0616faa-e5d3-4d0e-a74b-926f0067b4ff'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to send email. Use send_email_tool.', 'tool_calls': [{'id': 'fc_f2299b83-610f-4783-9997-eed0843dea5f', 'function': {'arguments': '{"body":"How are you?","recipient":"wrong@test.com","subject":"Hello"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 58, 'prompt_tokens': 175, 'total_tokens': 233, 'completion_time': 0.122868289, 'completion_tokens_details': {'reasoning_tokens': 12}, 'prompt_time': 0.008148588, 'prompt_tokens_details': None, 'queue_time': 0.318452818, 'total_time': 0.131016877}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_dee443f41b', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provide

In [63]:
# Step 2: Edit and Approve

from langgraph.types import Command

if "__interrupt__" in result:
    print("Paused! Approving...")
    
    result = agent.invoke(
    Command(
        resume={
            "decisions": [
                {
                    "type": "edit",
                    "edited_action": {          # snake_case in Python
                        "name": "send_email",   # same tool name
                        "args": {
                            "recipient": "safe@company.com",
                            "subject": "Meeting",
                            "body": "Updated message here",
                        },
                    },
                }
            ]
        }
    ),
    config=config,  # same thread_id
)

Paused! Approving...


KeyError: 'edited_action'

In [64]:
result

{'messages': [HumanMessage(content="Send email to wrong@test.com  with subject 'Hello' and Body 'How are you?'", additional_kwargs={}, response_metadata={}, id='c0616faa-e5d3-4d0e-a74b-926f0067b4ff'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to send email. Use send_email_tool.', 'tool_calls': [{'id': 'fc_f2299b83-610f-4783-9997-eed0843dea5f', 'function': {'arguments': '{"body":"How are you?","recipient":"wrong@test.com","subject":"Hello"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 58, 'prompt_tokens': 175, 'total_tokens': 233, 'completion_time': 0.122868289, 'completion_tokens_details': {'reasoning_tokens': 12}, 'prompt_time': 0.008148588, 'prompt_tokens_details': None, 'queue_time': 0.318452818, 'total_time': 0.131016877}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_dee443f41b', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provide